In [1]:
# Bag of Words:converting text into numerical features

from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Sample documents
documents = [
    "Data science is fun",
    "NLP is a part of data science",
    "Students learn NLP with Python",
    "Python is useful for data science"
]

# Create CountVectorizer object
vectorizer = CountVectorizer()

# Convert text documents into Bag of Words matrix
bow_matrix = vectorizer.fit_transform(documents)

# Get vocabulary / feature names
feature_names = vectorizer.get_feature_names_out()

# Convert matrix into a readable table
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=feature_names
)

# Display result
print("Vocabulary:")
print(feature_names)

print("\nBag of Words Matrix:")
print(bow_df)

Vocabulary:
['data' 'for' 'fun' 'is' 'learn' 'nlp' 'of' 'part' 'python' 'science'
 'students' 'useful' 'with']

Bag of Words Matrix:
   data  for  fun  is  learn  nlp  of  part  python  science  students  \
0     1    0    1   1      0    0   0     0       0        1         0   
1     1    0    0   1      0    1   1     1       0        1         0   
2     0    0    0   0      1    1   0     0       1        0         1   
3     1    1    0   1      0    0   0     0       1        1         0   

   useful  with  
0       0     0  
1       0     0  
2       0     1  
3       1     0  


In [ ]:
# Bag of Words:converting text into numerical features

from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Sample documents
documents = [
    "Data science is fun",
    "NLP is a part of data science",
    "Students learn NLP with Python",
    "Python is useful for data science"
]

# Create CountVectorizer object
vectorizer = CountVectorizer()

# Convert text documents into Bag of Words matrix
bow_matrix = vectorizer.fit_transform(documents)

# Get vocabulary / feature names
feature_names = vectorizer.get_feature_names_out()

# Convert matrix into a readable table
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=feature_names
)

# Display result
print("Vocabulary:")
print(feature_names)

print("\nBag of Words Matrix:")
print(bow_df)

In [3]:
%pip install Gensim

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
    --------------------------------------- 0.5/24.4 MB 2.9 MB/s eta 0:00:09
   - -------------------------------------- 1.0/24.4 MB 2.8 MB/s eta 0:00:09
   -- ------------------------------------- 1.3/24.4 MB 2.8 MB/s eta 0:00:09
   --- ------------------------------------ 2.1/24.4 MB 2.9 MB/s eta 0:00:08
   ----- ---------------------------------- 3.4/24.4 MB 3.4 MB/s eta 0:00:07
   ------ --------------------------------- 3.9/24.4 MB 3.5 MB/s eta 0:00:06
   ------- -------------------------------- 4.5/24.4 MB 3.3 MB/s eta 0:00:07
   --------- ------------------------------ 5.5/24.4 MB 3.3 MB/s eta 0:00:06
   ---------- ----------------------------- 6.3/24.4 MB 3.4 MB/s eta 0:00:06
   ----------- ---------------------------- 7.1/24.4 MB 3.5 MB/s eta 0:00:05
   ------------ --------------------------- 7.9/24.4 MB 3.6 MB/s eta 0:00:05
   --


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Anushka\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as api

# 1. Sample Data
job_description = "Looking for a frontend software engineer with experience in React and JavaScript."
resume_1 = "Frontend developer skilled in JS and React.js." # Semantic match, different words
resume_2 = "Experienced backend engineer using Java and SQL." # Poor match

documents = [job_description, resume_1, resume_2]

# ==========================================
# APPROACH A: TF-IDF (Exact Keyword Matching)
# ==========================================
print("--- TF-IDF Results ---")
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

# Compare Job Desc (Index 0) to Resume 1 (Index 1) and Resume 2 (Index 2)
tfidf_sim_1 = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]
tfidf_sim_2 = cosine_similarity(tfidf_matrix[0], tfidf_matrix[2])[0][0]

print(f"Similarity with Resume 1 (TF-IDF): {tfidf_sim_1:.3f}")
# Notice TF-IDF scores Resume 1 poorly because "engineer" != "developer" and "JavaScript" != "JS"
print(f"Similarity with Resume 2 (TF-IDF): {tfidf_sim_2:.3f}\n")


# ==========================================
# APPROACH B: FastText / Word2Vec (Semantic Matching)
# ==========================================
print("--- Dense Embedding Results ---")
# Note: For production, you would train a custom FastText model on your HR data.
# For this script, we are loading a lightweight pre-trained GloVe/Word2Vec model for demonstration.
try:
    # Loading a small pre-trained model via Gensim (glove-twitter-25 is fast for testing)
    model = api.load("glove-twitter-25") 

    def get_document_vector(text, model):
        """Averages the word vectors in a document to create a single document vector."""
        words = text.lower().split()
        # FastText handles OOV automatically, but for standard Word2Vec/GloVe we must filter
        valid_words = [word for word in words if word in model] 

        if not valid_words:
            return np.zeros(model.vector_size)

        # Average the vectors of all words in the document
        return np.mean([model[word] for word in valid_words], axis=0)

    # Vectorize documents
    vec_job = get_document_vector(job_description, model)
    vec_res1 = get_document_vector(resume_1, model)
    vec_res2 = get_document_vector(resume_2, model)

    # Calculate Cosine Similarity (Requires reshaping for sklearn)
    dense_sim_1 = cosine_similarity(vec_job.reshape(1, -1), vec_res1.reshape(1, -1))[0][0]
    dense_sim_2 = cosine_similarity(vec_job.reshape(1, -1), vec_res2.reshape(1, -1))[0][0]

    print(f"Similarity with Resume 1 (Semantic): {dense_sim_1:.3f}")
    # Semantic scoring understands Resume 1 is a much better fit, despite differing vocabulary
    print(f"Similarity with Resume 2 (Semantic): {dense_sim_2:.3f}")

except Exception as e:
    print("Could not load embedding model. Ensure you have internet access to download the Gensim model.")
 

--- TF-IDF Results ---
Similarity with Resume 1 (TF-IDF): 0.181
Similarity with Resume 2 (TF-IDF): 0.102

--- Dense Embedding Results ---
[==================================================] 100.0% 104.8/104.8MB downloaded
Similarity with Resume 1 (Semantic): 0.954
Similarity with Resume 2 (Semantic): 0.923


In [6]:
# Available Job Positions
jobs = {
 
"AI Engineer":
"""
Large language models, machine learning,
deep learning, neural networks,
generative AI, retrieval augmented generation,
AI agents, model deployment and MLOps.
""",
 
"Data Engineer":
"""
Cloud infrastructure, ETL pipelines,
data lakes, distributed systems,
SQL, NoSQL, Spark, Airflow,
data architecture and scalability.
""",
 
"Business Intelligence Analyst":
"""
Dashboard design, KPIs,
Tableau, Power BI,
business reporting,
decision support and storytelling.
""",
 
"Sustainability Analyst":
"""
ESG reporting,
environmental monitoring,
sustainability metrics,
climate risk assessment
and environmental policy analysis.
""",
 
"Supply Chain Analyst":
"""
Demand forecasting,
inventory optimization,
transportation planning,
operations research,
logistics analytics and procurement.
""",
 
"Biomedical AI Specialist":
"""
Medical imaging,
diagnostic systems,
computer vision,
healthcare analytics,
deep learning and radiology.
"""
}
 
 
# Profiles
applicants = {
 
"A1":
"""
I enjoy building conversational AI systems,
working with transformers and deploying
machine learning models.
""",
 
"A2":
"""
I have experience in cloud computing,
data warehousing and designing scalable
data infrastructures.
""",
 
"A3":
"""
I enjoy analyzing climate indicators,
environmental risks and sustainability reporting.
""",
 
"A4":
"""
I like creating dashboards and translating
complex business information into visual stories.
""",
 
"A5":
"""
I am interested in medical imaging,
computer vision and healthcare applications.
"""
}

In [14]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. The Job Data
jobs = {
    "AI Engineer": "Large language models, machine learning, deep learning, neural networks, generative AI, retrieval augmented generation, AI agents, model deployment and MLOps.",
    "Data Engineer": "Cloud infrastructure, ETL pipelines, data lakes, distributed systems, SQL, NoSQL, Spark, Airflow, data architecture and scalability.",
    "Business Intelligence Analyst": "Dashboard design, KPIs, Tableau, Power BI, business reporting, decision support and storytelling.",
    "Sustainability Analyst": "ESG reporting, environmental monitoring, sustainability metrics, climate risk assessment and environmental policy analysis.",
    "Supply Chain Analyst": "Demand forecasting, inventory optimization, transportation planning, operations research, logistics analytics and procurement.",
    "Biomedical AI Specialist": "Medical imaging, diagnostic systems, computer vision, healthcare analytics, deep learning and radiology."
}

job_titles = list(jobs.keys())
job_texts = list(jobs.values())

# 2. Initialize Vectorizer
vectorizer = TfidfVectorizer(stop_words='english')

# 3. Create the TF-IDF Matrix for the jobs
job_matrix = vectorizer.fit_transform(job_texts)

# 4. Extract the vocabulary words
feature_names = vectorizer.get_feature_names_out()

# 5. Convert the sparse matrix into a readable Pandas DataFrame
tfidf_df = pd.DataFrame(
    job_matrix.toarray(),
    columns=feature_names,
    index=job_titles
)

# Display the matrix (Rounding to 3 decimal places for readability)
print("TF-IDF Matrix (Showing the first 10 vocabulary words):")
# We use .iloc[:, :10] just to slice the first 10 columns so it fits nicely on the screen
print(tfidf_df.iloc[:, :10].round(3)) 

# To view a specific word's weight across all jobs, you can query the column directly:
print("\n--- Specific Word Weights for 'learning' ---")
print(tfidf_df['learning'].round(3))

TF-IDF Matrix (Showing the first 10 vocabulary words):
                               agents     ai  airflow  analysis  analytics  \
AI Engineer                     0.216  0.433    0.000      0.00      0.000   
Data Engineer                   0.000  0.000    0.245      0.00      0.000   
Business Intelligence Analyst   0.000  0.000    0.000      0.00      0.000   
Sustainability Analyst          0.000  0.000    0.000      0.27      0.000   
Supply Chain Analyst            0.000  0.000    0.000      0.00      0.251   
Biomedical AI Specialist        0.000  0.000    0.000      0.00      0.263   

                               architecture  assessment  augmented     bi  \
AI Engineer                           0.000        0.00      0.216  0.000   
Data Engineer                         0.245        0.00      0.000  0.000   
Business Intelligence Analyst         0.000        0.00      0.000  0.306   
Sustainability Analyst                0.000        0.27      0.000  0.000   
Supply Chain 

In [19]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# 1. Job Positions Data
jobs = {
    "AI Engineer": """Large language models, machine learning, deep learning, neural networks, generative AI, retrieval augmented generation, AI agents, model deployment and MLOps.""",
    "Data Engineer": """Cloud infrastructure, ETL pipelines, data lakes, distributed systems, SQL, NoSQL, Spark, Airflow, data architecture and scalability.""",
    "Business Intelligence Analyst": """Dashboard design, KPIs, Tableau, Power BI, business reporting, decision support and storytelling.""",
    "Sustainability Analyst": """ESG reporting, environmental monitoring, sustainability metrics, climate risk assessment and environmental policy analysis.""",
    "Supply Chain Analyst": """Demand forecasting, inventory optimization, transportation planning, operations research, logistics analytics and procurement.""",
    "Biomedical AI Specialist": """Medical imaging, diagnostic systems, computer vision, healthcare analytics, deep learning and radiology."""
}

# 2. Applicant Profiles Data
applicants = {
    "A1": """I enjoy building conversational AI systems, working with transformers and deploying machine learning models.""",
    "A2": """I have experience in cloud computing, data warehousing and designing scalable data infrastructures.""",
    "A3": """I enjoy analyzing climate indicators, environmental risks and sustainability reporting.""",
    "A4": """I like creating dashboards and translating complex business information into visual stories.""",
    "A5": """I am interested in medical imaging, computer vision and healthcare applications."""
}

# Setup labels and lists
job_names = list(jobs.keys())
applicant_ids = list(applicants.keys())
all_documents = list(jobs.values()) + list(applicants.values())
num_jobs = len(job_names)

# ==========================================
# 1. BAG OF WORDS (Direct Matrix Intersection)
# ==========================================
count_vectorizer = CountVectorizer(stop_words='english')
bow_matrix = count_vectorizer.fit_transform(all_documents).toarray()

bow_jobs = bow_matrix[:num_jobs]          # Shape: (6, 95)
bow_applicants = bow_matrix[num_jobs:]    # Shape: (5, 95)

# Matrix Multiplication calculates total shared word counts (Dot Product)
bow_score_matrix = np.dot(bow_applicants, bow_jobs.T)

# ==========================================
# 2. TF-IDF (Direct Combined Weights)
# ==========================================
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(all_documents).toarray()

tfidf_jobs = tfidf_matrix[:num_jobs]
tfidf_applicants = tfidf_matrix[num_jobs:]

# Element-wise multiplication of vectors captures only shared vocabulary weights
tfidf_score_matrix = np.zeros((len(applicant_ids), num_jobs))
for a_idx in range(len(applicant_ids)):
    for j_idx in range(num_jobs):
        # Multiply weights of matching words and sum them up
        overlap_weights = tfidf_applicants[a_idx] * tfidf_jobs[j_idx]
        tfidf_score_matrix[a_idx][j_idx] = np.sum(overlap_weights)


# ==========================================
# OUTPUT GENERATION
# ==========================================
print(f"{'Applicant':<12} | {'Best Match Job Role':<30} | {'BoW Intersect':<15} | {'TF-IDF Weight Sum'}")
print("-" * 78)

for i, app_id in enumerate(applicant_ids):
    # Determine the index with the highest combined intersection score
    best_job_idx = np.argmax(tfidf_score_matrix[i])
    matched_job = job_names[best_job_idx]
    
    bow_intersect = int(bow_score_matrix[i][best_job_idx])
    tfidf_weight = tfidf_score_matrix[i][best_job_idx]
    
    print(f"Applicant {app_id:<3} | {matched_job:<30} | {bow_intersect:<15} | {tfidf_weight:.3f}")

Applicant    | Best Match Job Role            | BoW Intersect   | TF-IDF Weight Sum
------------------------------------------------------------------------------
Applicant A1  | AI Engineer                    | 6               | 0.306
Applicant A2  | Data Engineer                  | 5               | 0.301
Applicant A3  | Sustainability Analyst         | 5               | 0.396
Applicant A4  | Business Intelligence Analyst  | 1               | 0.077
Applicant A5  | Biomedical AI Specialist       | 5               | 0.535


In [20]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# 1. Job Positions Data
jobs = {
    "AI Engineer": """Large language models, machine learning, deep learning, neural networks, generative AI, retrieval augmented generation, AI agents, model deployment and MLOps.""",
    "Data Engineer": """Cloud infrastructure, ETL pipelines, data lakes, distributed systems, SQL, NoSQL, Spark, Airflow, data architecture and scalability.""",
    "Business Intelligence Analyst": """Dashboard design, KPIs, Tableau, Power BI, business reporting, decision support and storytelling.""",
    "Sustainability Analyst": """ESG reporting, environmental monitoring, sustainability metrics, climate risk assessment and environmental policy analysis.""",
    "Supply Chain Analyst": """Demand forecasting, inventory optimization, transportation planning, operations research, logistics analytics and procurement.""",
    "Biomedical AI Specialist": """Medical imaging, diagnostic systems, computer vision, healthcare analytics, deep learning and radiology."""
}

# 2. Applicant Profiles Data
applicants = {
    "A1": """I enjoy building conversational AI systems, working with transformers and deploying machine learning models.""",
    "A2": """I have experience in cloud computing, data warehousing and designing scalable data infrastructures.""",
    "A3": """I enjoy analyzing climate indicators, environmental risks and sustainability reporting.""",
    "A4": """I like creating dashboards and translating complex business information into visual stories.""",
    "A5": """I am interested in medical imaging, computer vision and healthcare applications."""
}

# Setup tracking structures
job_names = list(jobs.keys())
applicant_ids = list(applicants.keys())
all_documents = list(jobs.values()) + list(applicants.values())
num_jobs = len(job_names)

# Initialize vectorizers and fit data
count_vectorizer = CountVectorizer(stop_words='english')
bow_matrix = count_vectorizer.fit_transform(all_documents).toarray()
feature_names = count_vectorizer.get_feature_names_out()

tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(all_documents).toarray()

# Segment matrices
bow_jobs, bow_applicants = bow_matrix[:num_jobs], bow_matrix[num_jobs:]
tfidf_jobs, tfidf_applicants = tfidf_matrix[:num_jobs], tfidf_matrix[num_jobs:]

# Calculate core matrix weight totals to establish the top match index
tfidf_score_matrix = np.dot(tfidf_applicants, tfidf_jobs.T)

# ==========================================
# EXTRACT KEY WORD IMPORTANCE PER APPLICANT
# ==========================================
print("=== KEYWORD IMPORTANCE BREAKDOWN ===")
print("-" * 85)

for i, app_id in enumerate(applicant_ids):
    # Find the top matched job index
    best_job_idx = np.argmax(tfidf_score_matrix[i])
    matched_job = job_names[best_job_idx]
    
    print(f"\n🚀 Applicant {app_id} -> Best Match: **{matched_job}**")
    
    # Isolate vector rows for this specific pair
    a_bow, j_bow = bow_applicants[i], bow_jobs[best_job_idx]
    a_tfidf, j_tfidf = tfidf_applicants[i], tfidf_jobs[best_job_idx]
    
    # Calculate intersecting metrics
    shared_bow = a_bow * j_bow
    shared_tfidf_weights = a_tfidf * j_tfidf
    
    # Extract index locations where an overlap actually occurred
    overlapping_indices = np.where(shared_tfidf_weights > 0)[0]
    
    # Collect information into a temporary breakdown list
    word_metrics = []
    for idx in overlapping_indices:
        word_metrics.append({
            'word': feature_names[idx],
            'count_match': int(a_bow[idx]), # Matches found in applicant text
            'tfidf_importance': shared_tfidf_weights[idx]
        })
        
    # Sort the words by their structural TF-IDF importance descending
    word_metrics = sorted(word_metrics, key=lambda x: x['tfidf_importance'], reverse=True)
    
    # Print individual word contribution tables
    print(f"   {'Word Token':<18} | {'BoW Occurrence':<15} | {'TF-IDF Pair Importance'}")
    print("   " + "-" * 58)
    for metric in word_metrics:
        print(f"   {metric['word']:<18} | {metric['count_match']:<15} | {metric['tfidf_importance']:.4f}")

=== KEYWORD IMPORTANCE BREAKDOWN ===
-------------------------------------------------------------------------------------

🚀 Applicant A1 -> Best Match: **AI Engineer**
   Word Token         | BoW Occurrence  | TF-IDF Pair Importance
   ----------------------------------------------------------
   ai                 | 1               | 0.1103
   learning           | 1               | 0.0853
   machine            | 1               | 0.0552
   models             | 1               | 0.0552

🚀 Applicant A2 -> Best Match: **Data Engineer**
   Word Token         | BoW Occurrence  | TF-IDF Pair Importance
   ----------------------------------------------------------
   data               | 2               | 0.2411
   cloud              | 1               | 0.0603

🚀 Applicant A3 -> Best Match: **Sustainability Analyst**
   Word Token         | BoW Occurrence  | TF-IDF Pair Importance
   ----------------------------------------------------------
   environmental      | 1               | 0.1660

In [21]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# 1. Job Positions Data
jobs = {
    "AI Engineer": """Large language models, machine learning, deep learning, neural networks, generative AI, retrieval augmented generation, AI agents, model deployment and MLOps.""",
    "Data Engineer": """Cloud infrastructure, ETL pipelines, data lakes, distributed systems, SQL, NoSQL, Spark, Airflow, data architecture and scalability.""",
    "Business Intelligence Analyst": """Dashboard design, KPIs, Tableau, Power BI, business reporting, decision support and storytelling.""",
    "Sustainability Analyst": """ESG reporting, environmental monitoring, sustainability metrics, climate risk assessment and environmental policy analysis.""",
    "Supply Chain Analyst": """Demand forecasting, inventory optimization, transportation planning, operations research, logistics analytics and procurement.""",
    "Biomedical AI Specialist": """Medical imaging, diagnostic systems, computer vision, healthcare analytics, deep learning and radiology."""
}

# 2. Applicant Profiles Data
applicants = {
    "A1": """I enjoy building conversational AI systems, working with transformers and deploying machine learning models.""",
    "A2": """I have experience in cloud computing, data warehousing and designing scalable data infrastructures.""",
    "A3": """I enjoy analyzing climate indicators, environmental risks and sustainability reporting.""",
    "A4": """I like creating dashboards and translating complex business information into visual stories.""",
    "A5": """I am interested in medical imaging, computer vision and healthcare applications."""
}

# Setup tracking structures
job_names = list(jobs.keys())
applicant_ids = list(applicants.keys())
all_documents = list(jobs.values()) + list(applicants.values())
num_jobs = len(job_names)

# Initialize vectorizers and fit data
count_vectorizer = CountVectorizer(stop_words='english')
bow_matrix = count_vectorizer.fit_transform(all_documents).toarray()
feature_names = count_vectorizer.get_feature_names_out()

tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(all_documents).toarray()

# Segment matrices
bow_jobs, bow_applicants = bow_matrix[:num_jobs], bow_matrix[num_jobs:]
tfidf_jobs, tfidf_applicants = tfidf_matrix[:num_jobs], tfidf_matrix[num_jobs:]

# Calculate core matrix weight totals to establish the top match index
tfidf_score_matrix = np.dot(tfidf_applicants, tfidf_jobs.T)

# ================================================================
# OUTPUT GENERATION: Matches with Complete Applicant TF-IDF Row Profiles
# ================================================================
print("=== APPLICANT MATCHES & FULL TF-IDF MATRIX PROFILES ===")
print("=" * 85)

for i, app_id in enumerate(applicant_ids):
    # Find the top matched job index based on dot product density
    best_job_idx = np.argmax(tfidf_score_matrix[i])
    matched_job = job_names[best_job_idx]
    
    print(f"\n👤 APPLICANT {app_id}")
    print(f"   🎯 Best Match Job Role : {matched_job}")
    print(f"   📊 Overlap Count (BoW) : {int(np.dot(bow_applicants[i], bow_jobs[best_job_idx].T))} words")
    print(f"   📈 Match Density Score : {tfidf_score_matrix[i][best_job_idx]:.4f}")
    
    print("\n   📋 COMPLETE APPLICANT TF-IDF MATRIX ROW (Non-Zero Weights):")
    print(f"      {'Vocabulary Token':<20} | {'TF-IDF Row Value'}")
    print("      " + "-" * 40)
    
    # Track down every word in this applicant's individual matrix vector row
    applicant_vector_row = tfidf_applicants[i]
    
    # Sort applicant tokens by their individual TF-IDF row weight
    sorted_word_indices = np.argsort(applicant_vector_row)[::-1]
    
    for idx in sorted_word_indices:
        weight = applicant_vector_row[idx]
        if weight > 0: # Print only tokens present in their resume profile
            print(f"      {feature_names[idx]:<20} | {weight:.4f}")
            
    print("\n" + "-" * 85)

=== APPLICANT MATCHES & FULL TF-IDF MATRIX PROFILES ===

👤 APPLICANT A1
   🎯 Best Match Job Role : AI Engineer
   📊 Overlap Count (BoW) : 6 words
   📈 Match Density Score : 0.3060

   📋 COMPLETE APPLICANT TF-IDF MATRIX ROW (Non-Zero Weights):
      Vocabulary Token     | TF-IDF Row Value
      ----------------------------------------
      working              | 0.3324
      transformers         | 0.3324
      conversational       | 0.3324
      building             | 0.3324
      deploying            | 0.3324
      models               | 0.2841
      ai                   | 0.2841
      machine              | 0.2841
      enjoy                | 0.2841
      learning             | 0.2498
      systems              | 0.2498

-------------------------------------------------------------------------------------

👤 APPLICANT A2
   🎯 Best Match Job Role : Data Engineer
   📊 Overlap Count (BoW) : 5 words
   📈 Match Density Score : 0.3014

   📋 COMPLETE APPLICANT TF-IDF MATRIX ROW (Non-Zero We